# Kernel Density Estimation (KDE) for Anomaly Detection

## What This Notebook Covers
This notebook implements **Kernel Density Estimation** for anomaly detection — both from scratch using NumPy and using scikit-learn's `KernelDensity` class. We estimate the probability density of normal data and flag low-density points as anomalies.

## Prerequisites
- Basic probability and statistics (PDF, Gaussian distribution)
- Familiarity with distance metrics (Euclidean distance)
- Python, NumPy, pandas, matplotlib

## Dataset
**Credit Card Fraud Detection** (Kaggle)  
Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
284,807 transactions with 30 features (V1-V28 from PCA + Time + Amount). Target: `Class` (0=normal, 1=fraud, ~0.17% fraud rate).

## Credits
- Dataset by ULB Machine Learning Group
- Parzen, E. (1962) — Original KDE paper
- Gradientts ML Intern Program — 2026

In [ ]:
# ============================================================
# Cell 2: All Imports
# ============================================================

import numpy as np                   # Core numerical computing
import pandas as pd                  # Data manipulation and loading
import matplotlib.pyplot as plt      # Plotting and visualisation
import seaborn as sns                # Statistical visualisation (built on matplotlib)
from sklearn.preprocessing import StandardScaler  # Feature scaling (zero mean, unit variance)
from sklearn.model_selection import train_test_split  # Train-test splitting
from sklearn.metrics import (        # Evaluation metrics for anomaly detection
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve
)
from sklearn.neighbors import KernelDensity  # Scikit-learn KDE implementation

# Reproducibility
np.random.seed(42)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports successful.')

## Part 1: Theory Recap

**Kernel Density Estimation (KDE)** is a non-parametric method to estimate the probability density function:

1. **Core idea:** Place a smooth "bump" (kernel) on every data point and sum them up to create a continuous density surface.
2. **Formula:** $\hat{f}(x) = \frac{1}{n \cdot h^d} \sum_{i=1}^{n} K\left(\frac{\|x - x_i\|}{h}\right)$ where $K$ is typically a Gaussian kernel.
3. **Bandwidth $h$** is the key hyperparameter — controls smoothness. Too small = overfitting (spiky). Too large = oversmoothing.
4. **Anomaly detection logic:** Points in low-density regions (small $\hat{f}(x)$) are flagged as anomalies.
5. **Limitation:** Scales poorly with dimensionality (curse of dimensionality) — density estimates become unreliable in high-d spaces.

## Loading the Dataset

We load the Credit Card Fraud Detection dataset. This is a real-world dataset with 284,807 transactions.
For the from-scratch implementation, we use a balanced subsample to keep computation tractable (KDE is O(n²)).

In [ ]:
# ============================================================
# Cell 4: Load the real-world dataset
# ============================================================

# Load the Credit Card Fraud Detection dataset
# Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
try:
    df = pd.read_csv('creditcard.csv')
    print('Loaded from local file.')
except FileNotFoundError:
    from sklearn.datasets import fetch_openml
    print('Local file not found. Downloading from OpenML...')
    credit = fetch_openml(data_id=1597, as_frame=True, parser='auto')
    df = credit.frame
    # OpenML may name the target differently
    if 'Class' not in df.columns:
        df = df.rename(columns={df.columns[-1]: 'Class'})
    df['Class'] = df['Class'].astype(int)
    print('Downloaded successfully.')

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:\n{df["Class"].value_counts()}')
print(f'\nFraud rate: {df["Class"].mean()*100:.3f}%')

print('\n--- First 5 rows ---')
df.head()

In [ ]:
# Quick exploration
print('--- Dataset Info ---')
print(df.info())
print('\n--- Statistical Summary ---')
df.describe()

## Preprocessing

The dataset has PCA-transformed features (V1-V28) which are already scaled, but `Time` and `Amount` need standardisation.
We create a balanced subsample for the from-scratch KDE (which is O(n²) and cannot handle 284K rows).

In [ ]:
# ============================================================
# Cell 5: Preprocessing
# ============================================================

# Step 1: Scale Time and Amount (V1-V28 are already PCA-scaled)
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = scaler.fit_transform(df[['Time']])

# Step 2: Drop original Time and Amount
df_clean = df.drop(['Time', 'Amount'], axis=1)

# Step 3: Check for nulls
print(f'Null values: {df_clean.isnull().sum().sum()}')

# Step 4: Create a manageable subsample for from-scratch KDE
# Take all fraud cases + a random sample of normal cases
fraud = df_clean[df_clean['Class'] == 1]
normal = df_clean[df_clean['Class'] == 0].sample(n=3000, random_state=42)
df_sub = pd.concat([normal, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(f'Subsample class distribution:\n{df_sub["Class"].value_counts()}')

# Step 5: Separate features and labels
X_sub = df_sub.drop('Class', axis=1).values
y_sub = df_sub['Class'].values

# Use select features for from-scratch (to manage dimensionality)
# KDE suffers from curse of dimensionality — use top features by variance
feature_cols = ['V1', 'V2', 'V3', 'V4', 'V10', 'V11', 'V14', 'V17', 'scaled_Amount']
X_selected = df_sub[feature_cols].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)

# For KDE training, use ONLY normal data (unsupervised anomaly detection)
X_train_normal = X_train[y_train == 0]

print(f'\nTraining set (normal only): {X_train_normal.shape}')
print(f'Test set: {X_test.shape} (fraud: {y_test.sum()}, normal: {(y_test==0).sum()})')

## Part 2: From Scratch Implementation

We implement KDE anomaly detection from scratch using only NumPy.

**What we are building:** A `KDEAnomalyDetector` class that:
1. Stores the training data (normal transactions)
2. Estimates density at any new point using the Gaussian kernel
3. Flags low-density points as anomalies

**Why from scratch?** Understanding the kernel computation, bandwidth's effect, and the O(n·m) scoring cost gives deep intuition for interview discussions about trade-offs.

In [ ]:
# ============================================================
# Cell 7: From-Scratch KDE Anomaly Detector (NumPy only)
# ============================================================

class KDEAnomalyDetector:
    """
    Kernel Density Estimation for anomaly detection.
    Uses Gaussian kernel to estimate density and flags
    low-density points as anomalies.
    """
    
    def __init__(self, bandwidth=1.0):
        # INTERVIEW NOTE: Bandwidth is the single most critical hyperparameter.
        # Too small = overfitting (every point is its own cluster)
        # Too large = oversmoothing (anomalies blend in)
        self.bandwidth = bandwidth
        self.X_train = None
    
    def _gaussian_kernel(self, distances):
        """
        Compute the Gaussian (RBF) kernel for given distances.
        K(u) = (1/sqrt(2*pi)) * exp(-u^2 / 2)
        """
        # INTERVIEW NOTE: Gaussian kernel ensures smooth, differentiable density estimate.
        # Other kernels (tophat, Epanechnikov) exist but Gaussian is most common.
        u = distances / self.bandwidth
        return (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * u ** 2)
    
    def fit(self, X):
        """
        Store training data. KDE is a lazy learner — no model is 'trained',
        the entire training set IS the model.
        """
        # INTERVIEW NOTE: KDE is memory-based. Training = O(1), but scoring = O(n*m).
        # This is the key trade-off vs. parametric models.
        self.X_train = np.array(X, dtype=np.float64)
        self.n_train, self.n_features = self.X_train.shape
        return self
    
    def score_samples(self, X):
        """
        Estimate log-density at each point in X.
        Returns log-density (not raw density) for numerical stability.
        """
        X = np.array(X, dtype=np.float64)
        n_test = X.shape[0]
        log_densities = np.zeros(n_test)
        
        for i in range(n_test):
            # Compute Euclidean distance from test point to ALL training points
            # INTERVIEW NOTE: This O(n) per query is why KDE doesn't scale well.
            distances = np.linalg.norm(self.X_train - X[i], axis=1)
            
            # Apply Gaussian kernel to each distance
            kernel_values = self._gaussian_kernel(distances)
            
            # Density = mean of kernel values, normalised by bandwidth^d
            # INTERVIEW NOTE: The h^d factor accounts for dimensionality.
            # In high-d, this normalisation causes density to shrink rapidly.
            density = np.mean(kernel_values) / (self.bandwidth ** self.n_features)
            
            # Log for numerical stability (densities can be extremely small)
            log_densities[i] = np.log(density + 1e-300)
        
        return log_densities
    
    def predict(self, X, contamination=0.01):
        """
        Predict anomaly labels: -1 = anomaly, 1 = normal.
        Uses the contamination parameter to set the threshold
        at the bottom percentile of density scores.
        """
        log_densities = self.score_samples(X)
        
        # INTERVIEW NOTE: The threshold is set as a percentile of scores.
        # contamination = expected fraction of anomalies.
        threshold = np.percentile(log_densities, contamination * 100)
        
        # Points with density below threshold are anomalies
        labels = np.where(log_densities < threshold, -1, 1)
        return labels
    
    @staticmethod
    def silverman_bandwidth(X):
        """
        Silverman's rule of thumb for bandwidth selection.
        h = 1.06 * sigma * n^(-1/5)
        """
        # INTERVIEW NOTE: This is a quick heuristic. For optimal bandwidth,
        # use cross-validated log-likelihood (GridSearchCV).
        n, d = X.shape
        sigma = np.mean(np.std(X, axis=0))  # Average std across features
        h = 1.06 * sigma * (n ** (-1.0 / (d + 4)))
        return h


print('KDEAnomalyDetector class defined successfully.')
print(f'Silverman bandwidth for training data: {KDEAnomalyDetector.silverman_bandwidth(X_train_normal):.4f}')

## Fitting and Evaluating the From-Scratch KDE

We fit the scratch KDE on **normal training data only** (unsupervised), then score the test set and evaluate against known fraud labels.

In [ ]:
# ============================================================
# Cell 8: Fit scratch model and evaluate
# ============================================================

# Use Silverman's bandwidth
bw = KDEAnomalyDetector.silverman_bandwidth(X_train_normal)
print(f'Using Silverman bandwidth: {bw:.4f}')

# Fit on normal data only
kde_scratch = KDEAnomalyDetector(bandwidth=bw)
kde_scratch.fit(X_train_normal)

# Score the test set
print('Scoring test set (this may take a moment)...')
scratch_log_densities = kde_scratch.score_samples(X_test)

# Predict with contamination matching the true fraud rate
contamination = y_test.mean()  # Use actual fraud rate
scratch_predictions = kde_scratch.predict(X_test, contamination=contamination)

# Convert predictions: -1 -> 1 (fraud), 1 -> 0 (normal) for comparison with y_test
scratch_pred_labels = np.where(scratch_predictions == -1, 1, 0)

print('\n--- From-Scratch KDE Results ---')
print(classification_report(y_test, scratch_pred_labels, target_names=['Normal', 'Fraud']))

# AUC-ROC using density scores (lower density = more anomalous = higher fraud score)
scratch_auc = roc_auc_score(y_test, -scratch_log_densities)  # Negate: lower density = higher score
print(f'AUC-ROC: {scratch_auc:.4f}')

scratch_ap = average_precision_score(y_test, -scratch_log_densities)
print(f'Average Precision (PR-AUC): {scratch_ap:.4f}')

## Part 3: Sklearn Implementation

Scikit-learn's `KernelDensity` uses **Ball-tree** or **KD-tree** data structures for efficient density queries — much faster than our O(n²) scratch implementation. It also provides `score_samples()` which returns **log-density** directly.

Key differences from our scratch version:
- Tree-based acceleration → O(d·log n) per query instead of O(n·d)
- Supports multiple kernels (gaussian, tophat, epanechnikov, etc.)
- Can use `GridSearchCV` with log-likelihood for bandwidth selection

In [ ]:
# ============================================================
# Cell 10: Sklearn KDE Implementation
# ============================================================

# Fit sklearn KDE on normal training data
sklearn_kde = KernelDensity(kernel='gaussian', bandwidth=bw)
sklearn_kde.fit(X_train_normal)

# Score the test set
sklearn_log_densities = sklearn_kde.score_samples(X_test)

# Apply threshold based on contamination
threshold = np.percentile(sklearn_log_densities, contamination * 100)
sklearn_pred_labels = np.where(sklearn_log_densities < threshold, 1, 0)

print('--- Sklearn KDE Results ---')
print(classification_report(y_test, sklearn_pred_labels, target_names=['Normal', 'Fraud']))

sklearn_auc = roc_auc_score(y_test, -sklearn_log_densities)
print(f'AUC-ROC: {sklearn_auc:.4f}')

sklearn_ap = average_precision_score(y_test, -sklearn_log_densities)
print(f'Average Precision (PR-AUC): {sklearn_ap:.4f}')

# --- Direct Comparison ---
print('\n=== Scratch vs Sklearn Comparison ===')
print(f'{"Metric":<25} {"Scratch":<15} {"Sklearn":<15}')
print(f'{"AUC-ROC":<25} {scratch_auc:<15.4f} {sklearn_auc:<15.4f}')
print(f'{"Avg Precision (PR-AUC)":<25} {scratch_ap:<15.4f} {sklearn_ap:<15.4f}')

## Visualisations

We create two key visualisations:
1. **Density score distribution** — comparing normal vs. fraud density scores (should show fraud at lower densities)
2. **ROC Curve** — comparing scratch vs. sklearn performance

In [ ]:
# ============================================================
# Cell 11: Visualisations
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Density Score Distribution ---
ax1 = axes[0]
ax1.hist(sklearn_log_densities[y_test == 0], bins=50, alpha=0.7, label='Normal', color='steelblue', density=True)
ax1.hist(sklearn_log_densities[y_test == 1], bins=50, alpha=0.7, label='Fraud', color='crimson', density=True)
ax1.axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.1f})')
ax1.set_xlabel('Log-Density Score', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('KDE Log-Density Distribution: Normal vs Fraud', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# --- Plot 2: ROC Curve (Scratch vs Sklearn) ---
ax2 = axes[1]
fpr_s, tpr_s, _ = roc_curve(y_test, -scratch_log_densities)
fpr_sk, tpr_sk, _ = roc_curve(y_test, -sklearn_log_densities)
ax2.plot(fpr_s, tpr_s, label=f'Scratch (AUC={scratch_auc:.3f})', color='darkorange', linewidth=2)
ax2.plot(fpr_sk, tpr_sk, label=f'Sklearn (AUC={sklearn_auc:.3f})', color='steelblue', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve: Scratch vs Sklearn KDE', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

# --- Plot 3: Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))
prec, rec, _ = precision_recall_curve(y_test, -sklearn_log_densities)
ax.plot(rec, prec, color='steelblue', linewidth=2, label=f'KDE (AP={sklearn_ap:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve (KDE Anomaly Detection)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

The most important hyperparameters for KDE anomaly detection are:

1. **Bandwidth (`h`)** — controls the smoothness of the density estimate. This is the single most impactful parameter.
2. **Kernel type** — Gaussian is standard, but tophat and Epanechnikov are alternatives.

We vary both and measure their effect on AUC-ROC.

In [ ]:
# ============================================================
# Cell 13: Hyperparameter Experiments
# ============================================================

# --- Experiment 1: Bandwidth Effect ---
bandwidths = [0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
bw_aucs = []

for bw_val in bandwidths:
    kde_exp = KernelDensity(kernel='gaussian', bandwidth=bw_val)
    kde_exp.fit(X_train_normal)
    scores = kde_exp.score_samples(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    bw_aucs.append(auc_val)
    print(f'Bandwidth={bw_val:<5.1f}  AUC-ROC={auc_val:.4f}')

# --- Experiment 2: Kernel Type Effect ---
kernels = ['gaussian', 'tophat', 'epanechnikov', 'exponential']
kernel_aucs = []
best_bw = bandwidths[np.argmax(bw_aucs)]

for kern in kernels:
    kde_exp = KernelDensity(kernel=kern, bandwidth=best_bw)
    kde_exp.fit(X_train_normal)
    scores = kde_exp.score_samples(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    kernel_aucs.append(auc_val)
    print(f'Kernel={kern:<15}  AUC-ROC={auc_val:.4f}')

# --- Plot the results ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bandwidth effect
axes[0].plot(bandwidths, bw_aucs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(x=best_bw, color='crimson', linestyle='--', label=f'Best h={best_bw}')
axes[0].set_xlabel('Bandwidth (h)', fontsize=12)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('Effect of Bandwidth on KDE Performance', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_xscale('log')

# Kernel type effect
bars = axes[1].bar(kernels, kernel_aucs, color=['steelblue', 'coral', 'mediumseagreen', 'mediumpurple'])
axes[1].set_xlabel('Kernel Type', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title(f'Effect of Kernel Type (bandwidth={best_bw})', fontsize=13, fontweight='bold')
for bar, val in zip(bars, kernel_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Key Conceptual Question: *"Why does KDE fail in high dimensions, and how would you work around it?"*

**Answer Structure:**

1. **The Curse of Dimensionality:** In high-dimensional spaces, data becomes sparse. The volume of the kernel (which scales as $h^d$) grows exponentially, causing density estimates to shrink toward zero for all points — both normal and anomalous. The signal-to-noise ratio collapses.

2. **Practical Symptom:** With 30+ features, all points have roughly the same (near-zero) density, making it impossible to distinguish normal from anomalous.

3. **Workarounds:**
   - **Dimensionality reduction** (PCA, t-SNE, UMAP) before applying KDE — reduce to 5-10 meaningful dimensions.
   - **Feature selection** — use only the most discriminative features.
   - **Switch algorithms** — Isolation Forest and LOF handle higher dimensions better.
   - **Use product kernels** — compute 1D KDE per feature and multiply (assumes independence).

4. **When KDE *does* work well:** Low-to-moderate dimensions (d < 10), sufficient data (n >> 2^d), and when the true density has smooth structure.

## Key Takeaways

1. **KDE is non-parametric** — it makes no assumption about the data distribution, placing a smooth kernel on each point to estimate density. This flexibility is its greatest strength and its greatest weakness (curse of dimensionality).

2. **Bandwidth is everything** — too small overfits, too large oversmooths. Use Silverman's rule as a starting point, then cross-validate with log-likelihood.

3. **KDE is a lazy learner** — training is O(1) (just stores data), but scoring is O(n·d) per query. This makes it unsuitable for large-scale production but excellent for exploratory analysis.

4. **Best suited for low-dimensional, small-to-medium datasets** — for high-d or large-n problems, prefer Isolation Forest (fast, handles high-d) or LOF (local density comparison).

5. **Anomaly = low density** — the anomaly detection logic is elegantly simple: estimate the density surface, and flag points in the valleys. This makes KDE highly interpretable compared to tree-based methods.